# Q.4 – ML + Quantum + Parallel Computing
**Goal:** pick the best **5 out of 12** feature groups for a crop-leaf classifier (healthy / unhealthy) so it fits on a small edge device.

Plan:
1. **(a)** Train a small neural network (MLP) and find out how useful each feature group is.
2. **(b)** Turn "choose 5 groups" into a QUBO (an energy formula). Low energy = good choice.
3. **(c)** Let a quantum algorithm (VQE, on a simulator) search for the lowest energy.
4. **(d)** Run many VQE starting points at the same time using `joblib` and measure the speed-up.
5. **(e)** Conclusion.

## Step 0 – Imports and setup
Here we load the libraries we need and set a few fixed values:
- `SEED = 44` so the random numbers are the same every time.
- `N = 12` feature groups, and we must keep `K = 5` of them.
- `NAMES` gives each group a readable name (colour, texture, shape ...).

In [1]:
import itertools, time, warnings
import numpy as np
from collections import Counter
from joblib import Parallel, delayed
from scipy.optimize import minimize
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

warnings.filterwarnings("ignore")
SEED, N, K = 44, 12, 5
NAMES = ["colour_mean", "colour_hist", "texture_glcm", "texture_lbp", "shape_area",
         "shape_perim", "edge_density", "vein_pattern", "spot_count", "leaf_ratio",
         "brightness", "noise_level"]

## (a) Machine Learning – part 1: make the data
- We create **600 fake (synthetic) rows**, each with 24 numbers. Every 2 numbers form 1 *feature group*, so we get 12 groups.
- The first groups carry real information, the next ones are copies/mixes of them (redundant), and the last ones are just noise.
- Then we split the data **70% train / 30% test** and scale the numbers so the neural network learns easily.

In [2]:
# ---------------- (a) MACHINE LEARNING ----------------
# 12 groups x 2 features = 24 columns. Cols 0-7 informative, 8-13 redundant, 14-23 noise
X, y = make_classification(n_samples=600, n_features=24, n_informative=8, n_redundant=6,
                           n_clusters_per_class=2, class_sep=1.2, shuffle=False,
                           random_state=SEED)
groups = [[2 * g, 2 * g + 1] for g in range(N)]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
sc = StandardScaler().fit(Xtr)
Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

## (a) Machine Learning – part 2: train the small MLP
- We train a small MLP (one hidden layer of 16 neurons) from scikit-learn.
- We then check it on the test data and print three scores:
  - **Accuracy** – how many predictions were right overall.
  - **Precision** – when it says "unhealthy", how often is it right.
  - **Recall** – out of all truly unhealthy leaves, how many did it catch.

In [3]:
mlp = MLPClassifier(hidden_layer_sizes=(16,), max_iter=800, random_state=SEED).fit(Xtr, ytr)
pred = mlp.predict(Xte)
print("== (a) MLP ==")
print(f"Accuracy {accuracy_score(yte, pred):.3f} | Precision {precision_score(yte, pred):.3f} "
      f"| Recall {recall_score(yte, pred):.3f}")

== (a) MLP ==
Accuracy 0.861 | Precision 0.822 | Recall 0.922


## (a) Machine Learning – part 3: how good is each feature group?
To find the value of one group, we **shuffle (break) that group** in the test data and see how much the accuracy drops.
- Big drop → the group is very useful.
- No drop → the group is useless (noise or duplicate).

We repeat 30 times and take the average. These values (`score`) are the "predictions for each of the 12 feature groups".

In [4]:
# value of each group = drop in accuracy when that group's predictions are 'broken' (shuffled)
rng = np.random.default_rng(SEED)
base = accuracy_score(yte, pred)
score = np.zeros(N)
for g, cols in enumerate(groups):
    drops = []
    for _ in range(30):
        Xp = Xte.copy()
        Xp[:, cols] = Xp[rng.permutation(len(Xp))][:, cols]
        drops.append(base - accuracy_score(yte, mlp.predict(Xp)))
    score[g] = max(np.mean(drops), 0)
print("Group value (accuracy drop when removed):")
for g in range(N):
    print(f"  x{g:<2} {NAMES[g]:<13} {score[g]:.4f}")

Group value (accuracy drop when removed):
  x0  colour_mean   0.1007
  x1  colour_hist   0.0665
  x2  texture_glcm  0.1054
  x3  texture_lbp   0.0704
  x4  shape_area    0.0296
  x5  shape_perim   0.0039
  x6  edge_density  0.0491
  x7  vein_pattern  0.0000
  x8  spot_count    0.0000
  x9  leaf_ratio    0.0000
  x10 brightness    0.0000
  x11 noise_level   0.0017


## (b) QUBO – part 1: write the energy and build the Q matrix
The energy formula is:

**E(x) = − Σ sᵢxᵢ + λ Σ rᵢⱼ xᵢxⱼ + P (Σxᵢ − 5)²**

- `xᵢ` = 1 if group *i* is kept, 0 if dropped (12 yes/no choices).
- `sᵢ` = usefulness of group *i* from step (a) (we want this big, so it has a minus sign).
- `rᵢⱼ` = how similar groups *i* and *j* are. Keeping two similar groups costs extra energy.
- `P (Σxᵢ − 5)²` = the **penalty**. It is zero only when exactly 5 groups are kept, and gets large otherwise.

Because xᵢ² = xᵢ, everything fits in one matrix `Q`, so E(x) = xᵀQx + constant. The code below builds `Q` and a function `energy(x)`.

In [5]:
# ---------------- (b) QUBO ----------------
# E(x) = -sum s_i x_i + lam * sum_{i<j} r_ij x_i x_j + P (sum x_i - K)^2
# r_ij = |corr| between groups (redundancy); s_i from the MLP above
Xall = np.vstack([Xtr, Xte])
gm = np.array([Xall[:, c].mean(axis=1) for c in groups])
R = np.abs(np.corrcoef(gm)); np.fill_diagonal(R, 0)
s = score / score.max()          # scale to 0..1
lam = 0.15
P = 2.0                          # > best possible gain, so rule breaking never pays
Q = np.zeros((N, N))
for i in range(N):
    Q[i, i] = -s[i] + P * (1 - 2 * K)       # x_i^2 = x_i
    for j in range(i + 1, N):
        Q[i, j] = lam * R[i, j] + 2 * P
const = P * K * K

def energy(x):
    x = np.asarray(x, float)
    return x @ Q @ x + const

## (b) QUBO – part 2: brute-force check
There are only 2¹² = 4096 possible answers, so we can try **all of them** and pick the lowest energy. This gives the *true* best answer.
We also check that:
- the best answer keeps exactly 5 groups, and
- it is the same as the best answer among only the "keep 5" choices (so the penalty works).

In [6]:
allx = np.array(list(itertools.product([0, 1], repeat=N)))
E_all = np.einsum("bi,ij,bj->b", allx, Q, allx) + const
order = np.argsort(E_all)
best = allx[order[0]]
print("\n== (b) QUBO brute force over 4096 states ==")
print("Lowest energy:", round(E_all[order[0]], 4), "bits:", "".join(map(str, best)),
      "groups:", [NAMES[i] for i in range(N) if best[i]])
print("Picks exactly 5?", best.sum() == K)
valid = allx.sum(1) == K
bv = allx[valid][np.argmin(E_all[valid])]
print("Best among valid-5 states equals global best?", np.array_equal(bv, best))


== (b) QUBO brute force over 4096 states ==
Lowest energy: -3.4781 bits: 111100100000 groups: ['colour_mean', 'colour_hist', 'texture_glcm', 'texture_lbp', 'edge_density']
Picks exactly 5? True
Best among valid-5 states equals global best? True


## (c) Quantum Computing – part 1: build the VQE simulator
VQE is a quantum + classical loop:
1. A small quantum circuit with adjustable angles `theta` makes a quantum state over 12 qubits (each qubit = one yes/no choice).
2. We compute the **average energy** of that state.
3. A classical optimizer (COBYLA) changes the angles to make the energy lower. Repeat.

The circuit is a **hardware-efficient ansatz**: layers of RY rotations on every qubit, followed by CZ gates between neighbouring qubits (in a ring).
We simulate it with plain NumPy (no quantum hardware needed). `run_vqe(seed)` does one full VQE run from a random starting point.

In [7]:
# ---------------- (c) VQE (numpy statevector simulator) ----------------
LAYERS = 4
NP_ = N * (LAYERS + 1)
idx = np.arange(2 ** N)
bits = ((idx[:, None] >> (N - 1 - np.arange(N))) & 1)       # qubit 0 = MSB
cz_sign = np.ones(2 ** N)
for q in range(N):                                           # CZ ring
    cz_sign *= 1 - 2 * (bits[:, q] & bits[:, (q + 1) % N])

def ry_all(psi, th):
    psi = psi.reshape([2] * N)
    for q in range(N):
        c, s_ = np.cos(th[q] / 2), np.sin(th[q] / 2)
        a = np.take(psi, 0, axis=q); b = np.take(psi, 1, axis=q)
        psi = np.stack([c * a - s_ * b, s_ * a + c * b], axis=q)
    return psi.reshape(-1)

def state(theta):
    th = theta.reshape(LAYERS + 1, N)
    psi = np.zeros(2 ** N); psi[0] = 1
    for l in range(LAYERS):
        psi = ry_all(psi, th[l]) * cz_sign
    return ry_all(psi, th[LAYERS])

def vqe_cost(theta):
    p = state(theta) ** 2
    return float(p @ E_all)

def run_vqe(seed, maxiter=250):
    r = np.random.default_rng(seed)
    t0 = r.uniform(0, 2 * np.pi, NP_)
    res = minimize(vqe_cost, t0, method="COBYLA", options={"maxiter": maxiter})
    return res.fun, res.x

## (c) Quantum Computing – part 2: run VQE and read the answer
- We start VQE from 8 different random angles and keep the **best** run.
- We then "measure" the final state 4096 times (shots) and count which bit-string shows up most.
- We report the best bit-string, its energy, how often it appeared, and whether it matches brute force.

Note: VQE can get stuck in a local minimum, so it may find a *good* answer that is not exactly the best one. Compare with brute force above.

In [8]:
print("\n== (c) VQE ==")
runs = [run_vqe(sd, 1500) for sd in range(8)]      # 8 random starts, keep the best
fun, th = min(runs, key=lambda r: r[0])
print("Energies of 8 starts:", [round(r[0], 2) for r in runs])
p = state(th) ** 2
top = np.argsort(p)[::-1][:5]
print("Final VQE energy (expectation):", round(fun, 4))
print("Top states by probability:")
for t in top:
    print("  ", "".join(map(str, allx[t])), f"p={p[t]:.3f}", f"E={E_all[t]:.4f}")
rs = np.random.default_rng(SEED).choice(2 ** N, 4096, p=p / p.sum())
cnt = Counter(rs)
bs, bc = cnt.most_common(1)[0]
print(f"4096 shots: best bitstring {''.join(map(str, allx[bs]))}, E={E_all[bs]:.4f}, "
      f"seen {bc}/4096 ({bc/4096:.1%})")
print("Matches brute force:", np.array_equal(allx[bs], best))


== (c) VQE ==
Energies of 8 starts: [np.float64(-2.03), np.float64(-2.96), np.float64(-2.94), np.float64(-2.48), np.float64(-2.94), np.float64(-1.61), np.float64(-2.67), np.float64(-2.94)]
Final VQE energy (expectation): -2.9555
Top states by probability:
   101100100010 p=0.993 E=-2.9583
   101100100001 p=0.003 E=-2.9714
   101100100100 p=0.002 E=-2.9623
   001100100101 p=0.001 E=-2.0394
   101010100010 p=0.000 E=-2.4803
4096 shots: best bitstring 101100100010, E=-2.9583, seen 4069/4096 (99.3%)
Matches brute force: False


## (d) Parallel Computing – joblib
VQE runs from different starting angles do not depend on each other, so we can run them **at the same time**.
- We run 16 starting points using `joblib.Parallel` with **1, 2 and 4 workers** and time each.
- **Speed-up** = time with 1 worker ÷ time with n workers.
- **Efficiency** = speed-up ÷ number of workers (1.0 is perfect).

Important: this only gets faster if your computer really has several CPU cores. On a 1-core machine the extra workers just add overhead.

In [9]:
# ---------------- (d) PARALLEL ----------------
print("\n== (d) joblib.Parallel over 16 starting points ==")
seeds = list(range(16))
times = {}
for w in (1, 2, 4):
    t = time.perf_counter()
    out = Parallel(n_jobs=w)(delayed(run_vqe)(sd, 400) for sd in seeds)
    times[w] = time.perf_counter() - t
    bestE = min(o[0] for o in out)
    print(f"workers={w}: {times[w]:.2f}s  best E over grid={bestE:.4f}")
for w in (1, 2, 4):
    sp = times[1] / times[w]
    print(f"workers={w}: speed-up {sp:.2f}x, efficiency {sp / w:.2f}")


== (d) joblib.Parallel over 16 starting points ==
workers=1: 55.88s  best E over grid=-2.7731
workers=2: 48.58s  best E over grid=-2.7731
workers=4: 51.20s  best E over grid=-2.7731
workers=1: speed-up 1.00x, efficiency 1.00
workers=2: speed-up 1.15x, efficiency 0.58
workers=4: speed-up 1.09x, efficiency 0.27


## (e) Conclusion
1. Brute force gives the true best 5 groups: colour_mean, colour_hist, texture_glcm, texture_lbp and edge_density. VQE found a valid 5-group answer that is close, but check its energy against brute force above.
2. The penalty term makes "exactly 5" work in the QUBO, but it also makes the VQE search harder, so many random restarts help.
3. Running starting points in parallel helps only when there are enough real CPU cores.

*(Update the numbers above to match your own run.)*